# 🧪 Prompt Engineering Playground
### TinyLlama-1.1B · Hugging Face Transformers ·

---
**What this notebook covers:**
1. Install dependencies & load TinyLlama locally
2. Define 6 prompt engineering styles
3. Run experiments & compare outputs with scoring
4. Interactive chatbot loop


## 📦 Step 1 — Install Dependencies

In [ ]:
# Run once to install required packages
!pip install -q transformers torch accelerate

## 🔧 Step 2 — Imports & Configuration

In [1]:
import time, textwrap, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"

GEN_CONFIG = dict(
    max_new_tokens=256,
    do_sample=True,
    temperature=0.7,
    top_p=0.95,
    repetition_penalty=1.15,
)

print(f"PyTorch {torch.__version__} | Device: {DEVICE.upper()}")

PyTorch 2.2.2 | Device: CPU


## 🤖 Step 3 — Load TinyLlama (downloads ~2.2 GB on first run, then cached)

In [2]:
print(f"Loading {MODEL_ID} …")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

pipe = pipeline(
    "text-generation",
    model=MODEL_ID,
    tokenizer=tokenizer,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto",
)

print("✅ Model ready!")


def ask(prompt: str) -> tuple:
    """Send a raw prompt; return (response_text, elapsed_sec)."""
    messages  = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    t0  = time.perf_counter()
    out = pipe(formatted, **GEN_CONFIG)
    return out[0]["generated_text"][len(formatted):].strip(), round(time.perf_counter()-t0, 1)

Loading TinyLlama/TinyLlama-1.1B-Chat-v1.0 …


/Users/softwareengineer/PromptPlayground/venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


✅ Model ready!


## ✏️ Step 4 — Define 6 Prompt Engineering Styles

In [3]:
def zero_shot(topic):    return f"Explain {topic}."

def few_shot(topic):     return textwrap.dedent(f"""
    Here are two examples of clear explanations:

    Topic: photosynthesis
    Answer: Photosynthesis converts sunlight, water, and CO₂ into glucose
    and oxygen inside plant chloroplasts.

    Topic: gravity
    Answer: Gravity is an attractive force between masses — the larger
    the mass, the stronger the pull.

    Now explain: {topic}
    Answer:
""").strip()

def chain_of_thought(topic): return (
    f"Let's think step by step.\n\n"
    f"I want to understand: {topic}\n\n"
    f"Walk me through the reasoning one step at a time, "
    f"then give a clear final summary."
)

def role_persona(topic): return (
    f"You are an expert professor who makes complex topics crystal-clear "
    f"to beginners. A first-year student asks:\n\n"
    f'"{topic}"\n\n'
    f"Give a friendly, engaging explanation with an analogy."
)

def structured_output(topic): return (
    f"Explain {topic} using this structure:\n\n"
    f"**Definition:** (1 sentence)\n"
    f"**How it works:** (2-3 bullet points)\n"
    f"**Real-world example:** (1 sentence)\n"
    f"**Why it matters:** (1 sentence)"
)

def socratic(topic): return (
    f"Use the Socratic method to help me understand {topic}. "
    f"Start with a thought-provoking question, then provide a short insight "
    f"that leads naturally to the answer. Keep it conversational."
)

STYLES = [
    ("🎯 Zero-Shot",        zero_shot),
    ("📚 Few-Shot",         few_shot),
    ("🔗 Chain-of-Thought", chain_of_thought),
    ("🎭 Role / Persona",   role_persona),
    ("📋 Structured Output",structured_output),
    ("🧠 Socratic",         socratic),
]

print(f"Loaded {len(STYLES)} prompt styles.")

Loaded 6 prompt styles.


## 📊 Step 5 — Scoring Function

In [4]:
def score(response: str, elapsed: float) -> dict:
    words        = response.split()
    sentences    = [s for s in response.split(".") if s.strip()]
    has_list     = any(c in response for c in ["-", "•", "*", "1."])
    has_analogy  = any(w in response.lower() for w in ["like", "similar", "imagine", "think of"])
    unique_ratio = len(set(words)) / max(len(words), 1)

    total = (
        min(len(words) / 80, 1.0)           * 3.0 +  # length
        min(unique_ratio * 1.5, 1.0)        * 3.0 +  # variety
        (0.3 if has_list    else 0.0)        * 2.0 +  # structure
        (0.2 if has_analogy else 0.0)        * 1.0 +  # analogy
        max(0, 1.0 - (elapsed - 2) / 20)    * 1.0    # speed
    )

    return {
        "score"     : round(total, 2),
        "words"     : len(words),
        "sentences" : len(sentences),
        "has_list"  : has_list,
        "analogy"   : has_analogy,
        "time_s"    : elapsed,
    }

def bar(v, mx=10, w=20): return "█"*int(v/mx*w) + "░"*(w-int(v/mx*w))

print("Scoring helpers ready.")

Scoring helpers ready.


## 🧪 Step 6 — Run Full Experiment (all 6 styles on one topic)

In [5]:
# ✏️  Change this topic to anything you like!
TOPIC = "neural networks"

results = []

for name, builder in STYLES:
    prompt = builder(TOPIC)
    print(f"\nRunning {name} …", flush=True)
    response, elapsed = ask(prompt)
    m = score(response, elapsed)
    results.append({"style": name, "prompt": prompt, "response": response, "metrics": m})

    print(f"  ✅  {elapsed}s  |  {m['words']} words  |  score {m['score']}/10")

print("\n🎉 All styles complete!")


Running 🎯 Zero-Shot …
  ✅  716.3s  |  193 words  |  score 6.6/10

Running 📚 Few-Shot …
  ✅  546.4s  |  180 words  |  score 6.2/10

Running 🔗 Chain-of-Thought …
  ✅  396.6s  |  198 words  |  score 6.6/10

Running 🎭 Role / Persona …
  ✅  409.0s  |  194 words  |  score 6.2/10

Running 📋 Structured Output …
  ✅  383.3s  |  211 words  |  score 6.8/10

Running 🧠 Socratic …
  ✅  346.3s  |  181 words  |  score 6.6/10

🎉 All styles complete!


## 🖨️ Step 7 — Print Detailed Results

In [6]:
DIV = "─" * 60

for r in results:
    m = r["metrics"]
    print(f"\n{DIV}")
    print(f"  {r['style']}")
    print(f"{DIV}")
    print(f"\nPROMPT:\n{textwrap.indent(r['prompt'], '  ')}")
    print(f"\nRESPONSE:\n{textwrap.indent(r['response'], '  ')}")
    print(f"\nSCORE: {bar(m['score'])}  {m['score']}/10")
    print(f"Words: {m['words']}  |  Time: {m['time_s']}s  |  List: {m['has_list']}  |  Analogy: {m['analogy']}")


────────────────────────────────────────────────────────────
  🎯 Zero-Shot
────────────────────────────────────────────────────────────

PROMPT:
  Explain neural networks.

RESPONSE:
  Neural networks are a type of artificial intelligence (AI) algorithm that mimics the way our brains operate. They consist of layers of interconnected nodes, called neurons, which process information and make decisions based on input data. Neural networks can be used to solve problems in various fields such as image recognition, speech recognition, natural language processing, and machine learning.

  Here's how it works:

  1. Input data: The first step is to provide an input data set, typically consisting of images or other examples. Each example consists of a set of features (e.g., color, texture, shape), and one or more labels indicating what object is represented by each feature.

  2. Preprocessing: The next step is to preprocess the input data. This involves transforming it into a format that is e

## 🏆 Step 8 — Comparison Table + Winner

In [7]:
ranked = sorted(results, key=lambda r: r["metrics"]["score"], reverse=True)

print(f"{'Style':<26} {'Score':>5}  {'Bar':<22} {'Words':>5}  {'Time':>6}")
print("─" * 70)
for r in ranked:
    m = r["metrics"]
    print(f"{r['style']:<26} {m['score']:>5.1f}  {bar(m['score']):<22} {m['words']:>5}  {m['time_s']:>5.1f}s")

print(f"\n🏆  Winner: {ranked[0]['style']}  — Score: {ranked[0]['metrics']['score']}/10")

Style                      Score  Bar                    Words    Time
──────────────────────────────────────────────────────────────────────
📋 Structured Output          6.8  █████████████░░░░░░░     211  383.3s
🎯 Zero-Shot                  6.6  █████████████░░░░░░░     193  716.3s
🔗 Chain-of-Thought           6.6  █████████████░░░░░░░     198  396.6s
🧠 Socratic                   6.6  █████████████░░░░░░░     181  346.3s
📚 Few-Shot                   6.2  ████████████░░░░░░░░     180  546.4s
🎭 Role / Persona             6.2  ████████████░░░░░░░░     194  409.0s

🏆  Winner: 📋 Structured Output  — Score: 6.8/10


---
## 📝 Summary of Prompt Styles

| Style | Best For | Key Technique |
|---|---|---|
| 🎯 Zero-Shot | Quick answers, simple facts | Direct question |
| 📚 Few-Shot | Consistent format, pattern following | 2–3 worked examples |
| 🔗 Chain-of-Thought | Reasoning, math, logic | "Think step by step" |
| 🎭 Role / Persona | Tone control, domain depth | Expert role assignment |
| 📋 Structured Output | Reports, docs, comparisons | Format template |
| 🧠 Socratic | Teaching, exploration | Question-led discovery |

### 💡 Key Takeaways
- **Context is everything** — more context → better responses
- **Format instructions** dramatically improve consistency
- **Role prompts** shift tone and vocabulary depth
- **CoT** unlocks reasoning ability hidden in small models
- TinyLlama performs best with **clear, structured prompts**
